# CLIP-based Image Search (semantic / meaning)

Search and organize a folder of images by **meaning** using CLIP embeddings.
Supports ViT-B/32 (fast), ViT-L/14 (precise), and ViT-H/14 (best).
Handles up to ~60k images; uses FAISS automatically past 10k for fast search.

**Methods**
1. Text → image search
2. Image → image search (upload a query)
3. Multi-query with negatives
4. Concept axis (bipolar projection)
5. Clustering by meaning (HDBSCAN + auto-labels)
6. 2D semantic map (UMAP)
7. Odd-one-out within a cluster

Every search shows filenames under the thumbnails and offers a **Download as ZIP** button.


## 1. Install dependencies

In [ ]:
pip install torch torchvision open_clip_torch pillow numpy scikit-learn umap-learn hdbscan matplotlib tqdm ipywidgets faiss-cpu

## 2. Imports & config

In [ ]:
import os, hashlib, pickle, math, io, zipfile
from datetime import datetime
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import open_clip
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import ipywidgets as widgets
from IPython.display import display, FileLink

# === EDIT THIS ===
IMAGE_FOLDER = Path('./Pictures')
CACHE_DIR    = Path('./cache_clip')
DOWNLOAD_DIR = Path('./downloads')        # where ZIPs of search results land
CACHE_DIR.mkdir(exist_ok=True)
DOWNLOAD_DIR.mkdir(exist_ok=True)
FORCE_RECOMPUTE = False

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.tif', '.tiff'}

device = ('mps' if torch.backends.mps.is_available()
          else 'cuda' if torch.cuda.is_available()
          else 'cpu')
print('device:', device)


## Pre-flight — clean the image folder

Move anything that shouldn't be processed into a sibling `*_dump/` folder:
- files whose extension isn't in `IMG_EXTS`
- files PIL can't open (corrupt / truncated)
- exact duplicates (same bytes, MD5-matched) — keeps the first occurrence

Run once before the scan. Re-running is safe: the dump folder lives outside `IMAGE_FOLDER` so it's never re-scanned.


In [ ]:
# Pre-flight cleanup — moves bad files out of IMAGE_FOLDER so they're not processed.
import shutil

DUMP_DIR = IMAGE_FOLDER.parent / f'{IMAGE_FOLDER.name}_dump'
DUMP_DIR.mkdir(exist_ok=True)

def _safe_move(src, dest_dir):
    """Move src into dest_dir, suffixing the name if it already exists."""
    dest = dest_dir / src.name
    n = 1
    while dest.exists():
        dest = dest_dir / f'{src.stem}__{n}{src.suffix}'
        n += 1
    shutil.move(str(src), str(dest))
    return dest

def _file_md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for buf in iter(lambda: f.read(chunk), b''):
            h.update(buf)
    return h.hexdigest()

bad_ext, bad_open, dupes = [], [], []
seen = {}  # md5 -> kept path

all_files = sorted(p for p in IMAGE_FOLDER.rglob('*') if p.is_file())
print(f'scanning {len(all_files)} files in {IMAGE_FOLDER}/ ...')

for p in tqdm(all_files):
    # 1. wrong extension
    if p.suffix.lower() not in IMG_EXTS:
        bad_ext.append(_safe_move(p, DUMP_DIR))
        continue
    # 2. unreadable / corrupt (verify only checks the header — cheap)
    try:
        with Image.open(p) as im:
            im.verify()
    except Exception:
        bad_open.append(_safe_move(p, DUMP_DIR))
        continue
    # 3. exact duplicate
    h = _file_md5(p)
    if h in seen:
        dupes.append((_safe_move(p, DUMP_DIR), seen[h].name))
    else:
        seen[h] = p

print(f'\nmoved to {DUMP_DIR}/:')
print(f'  • {len(bad_ext):>4} non-image extensions')
print(f'  • {len(bad_open):>4} unreadable / corrupt files')
print(f'  • {len(dupes):>4} exact duplicates')
print(f'kept in {IMAGE_FOLDER}/: {len(seen)} unique images')
if dupes:
    print('\nfirst few duplicates (moved → kept):')
    for moved, kept in dupes[:5]:
        print(f'  {moved.name}  →  {kept}')


## 3. Choose & load CLIP model

- **ViT-B/32** — fast, 512-dim, ~150 MB. Good default.
- **ViT-L/14** — ~3× slower on encode, 768-dim, ~900 MB. Noticeably sharper semantics.
- **ViT-H/14** — ~6× slower, 1024-dim, ~2 GB. Best quality, RAM-hungry.

Cache is keyed by model name, so switching models won't collide — each model gets its own index file.


In [ ]:
MODEL_CHOICES = {
    'ViT-B/32  (fast)':             ('ViT-B-32', 'laion2b_s34b_b79k'),
    'ViT-L/14  (precise, 3× slower)': ('ViT-L-14', 'laion2b_s32b_b82k'),
    'ViT-H/14  (best, 6× slower)':    ('ViT-H-14', 'laion2b_s32b_b79k'),
}
model_dd = widgets.Dropdown(options=list(MODEL_CHOICES), description='Model:',
                            layout=widgets.Layout(width='50%'))
display(model_dd)
print('Pick a model, then run the next cell.')


In [ ]:
MODEL_NAME, PRETRAINED = MODEL_CHOICES[model_dd.value]

cache_dir_hf = os.path.abspath('./hf_cache')
os.makedirs(cache_dir_hf, exist_ok=True)

model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME, pretrained=PRETRAINED, cache_dir=cache_dir_hf)
model = model.to(device).eval()
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
print(f'loaded {MODEL_NAME} / {PRETRAINED} on {device}')

@torch.no_grad()
def encode_text(texts):
    if isinstance(texts, str): texts = [texts]
    tokens = tokenizer(texts).to(device)
    feats = model.encode_text(tokens).float()
    feats /= feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy()

@torch.no_grad()
def encode_images(pil_images):
    batch = torch.stack([preprocess(im) for im in pil_images]).to(device)
    feats = model.encode_image(batch).float()
    feats /= feats.norm(dim=-1, keepdim=True)
    return feats.cpu().numpy()


## 4. Scan folder + compute/cache embeddings

Cache key depends on folder contents **and** model name — switching models recomputes automatically, but re-running with the same model hits the cache.


In [ ]:
def list_images(folder):
    return sorted([p for p in Path(folder).rglob('*') if p.suffix.lower() in IMG_EXTS])

def cache_key(paths):
    h = hashlib.md5()
    for p in paths:
        h.update(str(p).encode())
        try: h.update(str(p.stat().st_size).encode())
        except FileNotFoundError: pass
    h.update(MODEL_NAME.encode())
    return h.hexdigest()[:16]

def build_index(folder, batch_size=32):
    paths = list_images(folder)
    if not paths:
        raise RuntimeError(f'No images found in {folder}')
    key = cache_key(paths)
    cache_file = CACHE_DIR / f'clip_{MODEL_NAME}_{key}.pkl'
    if cache_file.exists() and not FORCE_RECOMPUTE:
        print(f'loading cached index: {cache_file.name}')
        with open(cache_file, 'rb') as f:
            return pickle.load(f)
    print(f'computing embeddings for {len(paths)} images with {MODEL_NAME}...')
    embs, valid = [], []
    for i in tqdm(range(0, len(paths), batch_size)):
        chunk = paths[i:i+batch_size]
        imgs = []
        for p in chunk:
            try:
                imgs.append(Image.open(p).convert('RGB'))
                valid.append(p)
            except Exception as e:
                print('skip', p, e)
        if imgs:
            embs.append(encode_images(imgs))
    index = {'paths': valid, 'embeddings': np.vstack(embs).astype(np.float32)}
    with open(cache_file, 'wb') as f:
        pickle.dump(index, f)
    print(f'cached to {cache_file}')
    return index

index = build_index(IMAGE_FOLDER)
paths      = index['paths']
embeddings = index['embeddings']
print('images:', len(paths), '| embedding dim:', embeddings.shape[1])


## 5. Search backend — brute-force or FAISS

Under 10k images, brute-force `embeddings @ q` is fastest and simplest.
Past that, FAISS `IndexFlatIP` gives the same exact cosine results with threaded search.


In [ ]:
USE_FAISS = len(paths) > 10_000

if USE_FAISS:
    import faiss
    d = embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(d)
    faiss_index.add(embeddings)
    print(f'FAISS IndexFlatIP: {faiss_index.ntotal} vectors, dim={d}')

    def search(query_vec, top_k=10):
        q = np.asarray(query_vec, dtype=np.float32).reshape(1, -1)
        scores, idx = faiss_index.search(q, top_k)
        return idx[0], scores[0]
else:
    print(f'brute-force search ({len(paths)} images)')
    def search(query_vec, top_k=10):
        sims = embeddings @ query_vec
        idx = np.argsort(-sims)[:top_k]
        return idx, sims[idx]


## 6. Display + download helpers

`show_grid` accepts `progress=True` to show a tqdm bar while loading thumbnails (useful for image-upload search and large Top-K).
`make_download_button(get_paths)` returns a button that zips the current result paths into `./downloads/` and prints a download link.


In [ ]:
def _short(name, n=22):
    """Truncate a filename for plot titles."""
    return name if len(name) <= n else name[:n-1] + '…'

def show_grid(image_paths, scores=None, cols=5, thumb=180, figtitle=None,
              show_filenames=True, progress=False):
    n = len(image_paths)
    if n == 0:
        print('(no results)'); return
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*thumb/72, rows*thumb/72))
    axes = np.array(axes).reshape(-1)
    for ax in axes: ax.axis('off')
    iterator = tqdm(enumerate(image_paths), total=n, desc='loading thumbs', leave=False) \
               if progress else enumerate(image_paths)
    for i, p in iterator:
        try:
            im = Image.open(p); im.thumbnail((thumb*2, thumb*2))
            axes[i].imshow(im)
            parts = []
            if scores is not None and i < len(scores):
                parts.append(f'{scores[i]:+.3f}')
            if show_filenames:
                parts.append(_short(Path(p).name))
            if parts:
                axes[i].set_title('\n'.join(parts), fontsize=7)
        except Exception as e:
            axes[i].set_title(f'err: {e}', fontsize=7)
    if figtitle: fig.suptitle(figtitle, fontsize=11)
    plt.tight_layout(); plt.show()

def _safe_prefix(s):
    """Sanitize a score/label string for use in a filename."""
    # keep digits, dot, plus, minus; replace everything else with underscore
    return ''.join(c if c.isalnum() or c in '.+-' else '_' for c in str(s))

def zip_results(paths_list, tag='results', prefixes=None):
    """Zip the given file paths into ./downloads/ and return the zip path.

    If `prefixes` is provided (same length as paths_list), each archived
    filename is prefixed with the corresponding string — e.g. prefix='0.543'
    on 'cat.png' becomes '0.543cat.png' inside the zip.
    """
    if not paths_list:
        print('nothing to download'); return None
    if prefixes is not None and len(prefixes) != len(paths_list):
        raise ValueError('prefixes must match paths_list in length')
    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    safe_tag = ''.join(c if c.isalnum() or c in '-_' else '_' for c in tag)[:40]
    zip_path = DOWNLOAD_DIR / f'{safe_tag}_{stamp}.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_STORED) as z:
        seen = {}
        for i, p in enumerate(paths_list):
            base = Path(p).name
            if prefixes is not None:
                base = f'{_safe_prefix(prefixes[i])}{base}'
            # disambiguate duplicate filenames from different folders
            if base in seen:
                seen[base] += 1
                stem, suf = Path(base).stem, Path(base).suffix
                arc = f'{stem}_{seen[base]}{suf}'
            else:
                seen[base] = 0
                arc = base
            z.write(p, arcname=arc)
    return zip_path

def make_download_button(get_paths, get_tag=lambda: 'results',
                         get_prefixes=lambda: None,
                         label='Download as ZIP'):
    btn = widgets.Button(description=label, icon='download')
    out = widgets.Output()
    def _on(_):
        out.clear_output()
        with out:
            zp = zip_results(get_paths(), tag=get_tag(), prefixes=get_prefixes())
            if zp is not None:
                print(f'{len(get_paths())} files → {zp}  ({zp.stat().st_size/1e6:.1f} MB)')
                display(FileLink(str(zp)))
                print(f'\n⚠️  Reminder: delete {zp} from ./downloads/ once you have saved it elsewhere — '
                      f'the folder fills up fast.')
    btn.on_click(_on)
    return widgets.VBox([btn, out])


## Method 1 — Text → image search

In [ ]:
# state shared between the search button and the download button
text_state = {'paths': [], 'prefixes': [], 'query': ''}

query_box = widgets.Text(value='', placeholder='e.g. a misty forest at dawn',
                         description='Query:', layout=widgets.Layout(width='60%'))
topk_box  = widgets.IntSlider(value=10, min=1, max=50, description='Top K')
run_btn   = widgets.Button(description='Search', button_style='primary')
out       = widgets.Output()

def _run(_):
    out.clear_output()
    with out:
        q = query_box.value.strip()
        if not q: print('Please enter a query.'); return
        print('encoding query…')
        idx, scores = search(encode_text(q)[0], top_k=topk_box.value)
        text_state['paths']    = [paths[i] for i in idx]
        text_state['prefixes'] = [f'{s:.3f}_' for s in scores]
        text_state['query']    = q
        show_grid(text_state['paths'], scores=scores,
                  figtitle=f'text → image: "{q}"',
                  progress=topk_box.value >= 12)

run_btn.on_click(_run)
query_box.continuous_update = False

dl = make_download_button(lambda: text_state['paths'],
                          get_tag=lambda: f'text_{text_state["query"]}',
                          get_prefixes=lambda: text_state['prefixes'])
display(widgets.VBox([widgets.HBox([query_box, topk_box, run_btn]), out, dl]))


## Method 2 — Image → image search

In [ ]:
img_state = {'paths': [], 'prefixes': [], 'query_name': ''}

uploader = widgets.FileUpload(accept='image/*', multiple=False, description='Upload query')
topk_box = widgets.IntSlider(value=10, min=1, max=50, description='Top K')
run_btn  = widgets.Button(description='Find similar', button_style='primary')
out      = widgets.Output()

def _run(_):
    out.clear_output()
    with out:
        if not uploader.value: print('Please upload an image first.'); return
        item = uploader.value[0] if isinstance(uploader.value, tuple) else next(iter(uploader.value.values()))
        query_img = Image.open(io.BytesIO(item['content'])).convert('RGB')
        img_state['query_name'] = item.get('name', 'query')
        print('encoding query image…')
        q = encode_images([query_img])[0]
        print('searching…')
        idx, scores = search(q, top_k=topk_box.value)
        img_state['paths']    = [paths[i] for i in idx]
        img_state['prefixes'] = [f'{s:.3f}_' for s in scores]
        fig, ax = plt.subplots(figsize=(2.5, 2.5))
        ax.imshow(query_img); ax.axis('off')
        ax.set_title(f'QUERY\n{_short(img_state["query_name"])}', fontsize=8)
        plt.show()
        show_grid(img_state['paths'], scores=scores,
                  figtitle='most similar in folder', progress=True)

run_btn.on_click(_run)
dl = make_download_button(lambda: img_state['paths'],
                          get_tag=lambda: f'similar_{Path(img_state["query_name"]).stem}',
                          get_prefixes=lambda: img_state['prefixes'])
display(widgets.VBox([widgets.HBox([uploader, topk_box, run_btn]), out, dl]))


## Method 3 — Multi-query with negatives

In [ ]:
mq_state = {'paths': [], 'prefixes': [], 'tag': 'multiquery'}

pos_box  = widgets.Text(value='', placeholder='comma-separated, e.g. city street, architecture',
                        description='Positive:', layout=widgets.Layout(width='70%'))
neg_box  = widgets.Text(value='', placeholder='comma-separated, e.g. people, cars',
                        description='Negative:', layout=widgets.Layout(width='70%'))
topk_box = widgets.IntSlider(value=10, min=1, max=50, description='Top K')
run_btn  = widgets.Button(description='Search', button_style='primary')
out      = widgets.Output()

def _run(_):
    out.clear_output()
    with out:
        positives = [s.strip() for s in pos_box.value.split(',') if s.strip()]
        negatives = [s.strip() for s in neg_box.value.split(',') if s.strip()]
        if not positives: print('Please enter at least one positive term.'); return
        print('encoding terms…')
        pos = encode_text(positives).mean(axis=0)
        vec = pos - encode_text(negatives).mean(axis=0) if negatives else pos
        vec /= np.linalg.norm(vec) + 1e-9
        idx, scores = search(vec.astype(np.float32), top_k=topk_box.value)
        mq_state['paths']    = [paths[i] for i in idx]
        mq_state['prefixes'] = [f'{s:+.3f}_' for s in scores]
        title = '+'.join(positives) + (' − ' + '−'.join(negatives) if negatives else '')
        mq_state['tag'] = title.replace(' ', '_')
        show_grid(mq_state['paths'], scores=scores, figtitle=title,
                  progress=topk_box.value >= 12)

run_btn.on_click(_run)
dl = make_download_button(lambda: mq_state['paths'],
                          get_tag=lambda: mq_state['tag'],
                          get_prefixes=lambda: mq_state['prefixes'])
display(widgets.VBox([pos_box, neg_box, widgets.HBox([topk_box, run_btn]), out, dl]))


## Method 4 — Concept axis

In [ ]:
axis_state = {'paths': [], 'tag': 'axis', 'scores': [], 'prefixes': []}

def concept_axis(pos_prompt, neg_prompt):
    v = encode_text(pos_prompt)[0] - encode_text(neg_prompt)[0]
    v /= np.linalg.norm(v) + 1e-9
    return embeddings @ v   # single vector × N — brute force is fine

neg_box  = widgets.Text(value='dark night',      description='Negative:', layout=widgets.Layout(width='60%'))
pos_box  = widgets.Text(value='bright daylight', description='Positive:', layout=widgets.Layout(width='60%'))
n_slider = widgets.IntSlider(value=12, min=2, max=30, description='# images')
run_btn  = widgets.Button(description='Show axis', button_style='primary')
out      = widgets.Output()

def _run(_):
    out.clear_output()
    with out:
        pos = pos_box.value.strip(); neg = neg_box.value.strip()
        if not pos or not neg: print('Please fill in both poles.'); return
        print('projecting…')
        proj = concept_axis(pos, neg)
        order = np.argsort(proj)
        picks = np.unique(np.linspace(0, len(order)-1, n_slider.value).astype(int))
        sel = [int(order[i]) for i in picks]
        axis_state['paths']    = [paths[i] for i in sel]
        axis_state['scores']   = [float(proj[i]) for i in sel]
        axis_state['prefixes'] = [f'{s:+.2f}_' for s in axis_state['scores']]
        axis_state['tag']      = f'axis_{neg}_to_{pos}'.replace(' ', '_')
        show_grid(axis_state['paths'], scores=axis_state['scores'],
                  cols=len(sel), thumb=140,
                  figtitle=f'{neg}  ←——————→  {pos}')

run_btn.on_click(_run)
dl = make_download_button(lambda: axis_state['paths'],
                          get_tag=lambda: axis_state['tag'],
                          get_prefixes=lambda: axis_state['prefixes'])
display(widgets.VBox([neg_box, pos_box, widgets.HBox([n_slider, run_btn]), out, dl]))


## Method 5 — Clustering by meaning (HDBSCAN + auto-labels)

After clustering, pick a cluster from the dropdown to download all its members as a ZIP.


In [ ]:
import hdbscan
from sklearn.preprocessing import normalize

DEFAULT_VOCAB = [
    'portrait of a person', 'landscape', 'forest', 'mountain', 'ocean', 'beach',
    'city street', 'building', 'interior of a room', 'food', 'animal', 'bird',
    'flower', 'machine', 'text document', 'abstract pattern', 'sky', 'night scene',
    'vehicle', 'crowd of people', 'artwork', 'diagram', 'close-up macro',
]

cluster_state = {
    'labels': np.array([]),
    'unique': [],
    'names':  {},   # cluster_id -> label name
}

vocab_area = widgets.Textarea(value='\n'.join(DEFAULT_VOCAB), description='Vocab:',
                              layout=widgets.Layout(width='70%', height='160px'))
min_size_slider = widgets.IntSlider(value=max(3, len(paths)//50), min=2,
                                    max=max(2, len(paths)//2), description='Min size')
sample_slider   = widgets.IntSlider(value=8, min=1, max=20, description='Samples/cluster')
show_outliers   = widgets.Checkbox(value=True, description='Show outliers')
run_btn         = widgets.Button(description='Cluster', button_style='primary')
out             = widgets.Output()

# download row — a dropdown of clusters + a button. Refreshed after each clustering run.
dl_dd  = widgets.Dropdown(options=[('— run clustering first —', None)], description='Cluster:')
dl_btn = widgets.Button(description='Download cluster as ZIP', icon='download')
dl_out = widgets.Output()

def _on_download(_):
    dl_out.clear_output()
    with dl_out:
        c = dl_dd.value
        if c is None: print('No cluster selected.'); return
        members = np.where(cluster_state['labels'] == c)[0]
        if c == -1:
            # outliers have no centroid — keep original order, no prefixes
            files    = [paths[i] for i in members]
            prefixes = None
            tag      = 'outliers'
        else:
            # rank by similarity to centroid (most prototypical first)
            sub = embeddings[members]
            centroid = sub.mean(axis=0)
            centroid /= np.linalg.norm(centroid) + 1e-9
            sims = sub @ centroid
            order = np.argsort(-sims)
            members = members[order]
            sims    = sims[order]
            files    = [paths[i] for i in members]
            prefixes = [f'{s:.3f}_' for s in sims]
            name     = cluster_state['names'].get(c, f'cluster_{c}')
            tag      = f'cluster{c}_{name}'.replace(' ', '_')
        zp = zip_results(files, tag=tag, prefixes=prefixes)
        if zp is not None:
            print(f'{len(files)} files → {zp}  ({zp.stat().st_size/1e6:.1f} MB)')
            display(FileLink(str(zp)))
            print(f'\n⚠️  Reminder: delete {zp} from ./downloads/ once you have saved it elsewhere — '
                  f'the folder fills up fast.')
dl_btn.on_click(_on_download)

def _run(_):
    out.clear_output()
    with out:
        vocab = [l.strip() for l in vocab_area.value.splitlines() if l.strip()]
        if not vocab: print('Please enter at least one vocab term.'); return
        print('encoding vocab…')
        vocab_emb = encode_text(vocab)
        print('clustering…')
        clusterer = hdbscan.HDBSCAN(min_cluster_size=min_size_slider.value, metric='euclidean')
        labels = clusterer.fit_predict(normalize(embeddings))
        unique = sorted(set(labels) - {-1})
        n_out = int((labels == -1).sum())
        cluster_state['labels'] = labels
        cluster_state['unique'] = unique
        cluster_state['names']  = {}
        print(f'{len(unique)} clusters, {n_out} outliers (of {len(labels)} images)')
        for c in unique:
            members  = np.where(labels == c)[0]
            centroid = embeddings[members].mean(axis=0)
            centroid /= np.linalg.norm(centroid) + 1e-9
            name = vocab[int(np.argmax(vocab_emb @ centroid))]
            cluster_state['names'][c] = name
            sample = members[:sample_slider.value]
            show_grid([paths[i] for i in sample],
                      cols=min(len(sample), 8), thumb=140,
                      figtitle=f'cluster {c} — "{name}" ({len(members)} images)')
        if show_outliers.value and n_out > 0:
            outs = np.where(labels == -1)[0][:sample_slider.value]
            show_grid([paths[i] for i in outs], cols=min(len(outs), 8), thumb=140,
                      figtitle=f'outliers ({n_out} total)')
        # refresh the download dropdown
        opts = [(f'cluster {c} — "{cluster_state["names"][c]}" ({int((labels==c).sum())})', c)
                for c in unique]
        if n_out > 0:
            opts.append((f'outliers ({n_out})', -1))
        dl_dd.options = opts if opts else [('— no clusters —', None)]

run_btn.on_click(_run)
display(widgets.VBox([
    vocab_area,
    widgets.HBox([min_size_slider, sample_slider]),
    widgets.HBox([show_outliers, run_btn]),
    out,
    widgets.HBox([dl_dd, dl_btn]),
    dl_out,
]))


## Method 6 — 2D semantic map (UMAP)

Reduce CLIP embeddings to 2D. Nearby = semantically similar.
For very large folders, consider subsampling — UMAP on 60k × 768 takes several minutes.


In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='umap')

import umap
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

max_umap = 5000
if len(paths) > max_umap:
    sample_idx = np.random.default_rng(0).choice(len(paths), size=max_umap, replace=False)
    print(f'subsampling {max_umap} of {len(paths)} for UMAP')
else:
    sample_idx = np.arange(len(paths))

reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
coords2d = reducer.fit_transform(embeddings[sample_idx])

fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(coords2d[:, 0], coords2d[:, 1], s=3, alpha=0.3, c='gray')
thumb_budget = min(250, len(sample_idx))
show_ids = np.random.default_rng(1).choice(len(sample_idx), size=thumb_budget, replace=False)
for j in show_ids:
    try:
        im = Image.open(paths[sample_idx[j]]); im.thumbnail((40, 40))
        ab = AnnotationBbox(OffsetImage(im, zoom=0.6), (coords2d[j, 0], coords2d[j, 1]), frameon=False)
        ax.add_artist(ab)
    except Exception: pass
ax.set_title(f'UMAP of {len(sample_idx)} CLIP embeddings'); ax.axis('off')
plt.tight_layout(); plt.show()


## Method 7 — Odd one out

For a given cluster, find members furthest from the centroid.
Run Method 5 first so cluster labels are populated.


In [ ]:
odd_state = {'paths': [], 'prefixes': [], 'cluster': None}

if len(cluster_state['unique']):
    cluster_dd = widgets.Dropdown(options=[int(c) for c in cluster_state['unique']],
                                  description='Cluster:')
    n_slider   = widgets.IntSlider(value=3, min=1, max=10, description='How many')
    run_btn    = widgets.Button(description='Find odd ones', button_style='primary')
    out        = widgets.Output()

    def _run(_):
        out.clear_output()
        with out:
            members = np.where(cluster_state['labels'] == cluster_dd.value)[0]
            sub = embeddings[members]
            centroid = sub.mean(axis=0)
            centroid /= np.linalg.norm(centroid) + 1e-9
            sims = sub @ centroid
            order = np.argsort(sims)
            sel_local = order[:n_slider.value]
            sel = [int(members[i]) for i in sel_local]
            sel_scores = [float(sims[i]) for i in sel_local]
            odd_state['paths']    = [paths[i] for i in sel]
            odd_state['prefixes'] = [f'{s:.3f}_' for s in sel_scores]
            odd_state['cluster']  = cluster_dd.value
            show_grid(odd_state['paths'], scores=sel_scores,
                      cols=min(len(sel), 6), thumb=160,
                      figtitle=f'odd ones out of cluster {cluster_dd.value}')

    run_btn.on_click(_run)
    dl = make_download_button(
        lambda: odd_state['paths'],
        get_tag=lambda: f'odd_cluster{odd_state["cluster"]}',
        get_prefixes=lambda: odd_state['prefixes'])
    display(widgets.VBox([widgets.HBox([cluster_dd, n_slider, run_btn]), out, dl]))
else:
    print('Run Method 5 (clustering) first.')
